# View and cluster structure

The `views` detector partitions a table's columns into dependence blocks
("views") and each view's rows into clusters, using closed-form marginal
likelihoods — then translates the partition into findings in the existing
taxonomy. It runs at standard and deep budgets only; the quick tier skips
it.

Every claim it emits carries the same honesty line: a greedy point
estimate, with scores that are marginal-likelihood values, not posterior
probabilities. There is no sampler behind it and no distribution over
partitions — one partition, chosen greedily, with its scores reported.


## Enabling it

Select the detector by id. With no selection, all detectors run; the
`views` findings appear alongside the exact passes (keys, functional
dependencies, gaps), which they complement rather than replace.


In [1]:
import onceover

ids = [d["id"] for d in onceover.available_detectors()]
print("views" in ids)


True


## Example: a shared regime

Two continuous columns share a mean shift at the same row; a third
column is independent noise. The pass should merge the shifting pair
into one view and leave the noise column singleton.


In [2]:
import random

import polars as pl

import onceover

rng = random.Random(3)
temperature = [20.0 + rng.gauss(0, 1) for _ in range(400)]
temperature += [32.0 + rng.gauss(0, 1) for _ in range(400)]
pressure = [1010.0 + rng.gauss(0, 5) for _ in range(400)]
pressure += [990.0 + rng.gauss(0, 5) for _ in range(400)]
humidity = [50.0 + rng.gauss(0, 10) for _ in range(800)]
df = pl.DataFrame(
    {"temperature": temperature, "pressure": pressure, "humidity": humidity}
)
report = onceover.profile(df, budget="standard", detectors=["views"])
for finding in report.findings:
    print(finding["category"], finding["subject"]["columns"], "::", finding["claim"])


changes ['humidity'] :: Rows split into 8 clusters; headline column humidity shifts by 2.47 between clusters (Greedy deterministic partition; scores are marginal-likelihood values, not posterior probabilities.).
changes ['pressure'] :: Rows split into 6 clusters; headline column pressure shifts by 1.57 between clusters (Greedy deterministic partition; scores are marginal-likelihood values, not posterior probabilities.).
changes ['temperature'] :: Rows split into 2 clusters; headline column temperature shifts by 1.00 between clusters (Greedy deterministic partition; scores are marginal-likelihood values, not posterior probabilities.).


## Reading the findings

Three finding kinds appear above. An `associations` / `statistical`
finding names a dependence block and its merge gain in nats — the
improvement of the joint view over the separate views, after the
per-cluster and per-view penalties. A `changes` / `residual_level`
finding names a view with two or more clusters and its largest
between-cluster shift. A `concentration` finding names spike cells:
individual cells whose log predictive density falls more than 10 nats
below their column's median, lowest first, capped at 20 per column.

A merge fires when a shared clustering matches or improves the
columns' singleton clusterings closely enough that the saved cluster
penalties outweigh any per-column marginal loss. That is what the
shared regime above provides: the joint clustering is the same split
both columns would choose alone. It is not a general dependence
detector — discrete columns whose singleton clusterings already reach
purity will not merge (see Limits).


## Example: spikes

A single column with three planted spikes. The spike cells rank first
by log predictive density; the finding also lists further low-density
cells under tight clusters, which is expected behavior rather than
additional spikes.


In [3]:
rng = random.Random(5)
series = [20.0 + rng.gauss(0, 1) for _ in range(200)]
for i in (30, 100, 170):
    series[i] = 999.9
df = pl.DataFrame({"temperature": series})
report = onceover.profile(df, budget="standard", detectors=["views"])
for finding in report.findings:
    print(finding["category"], finding["subject"]["columns"], "::", finding["claim"])


changes ['temperature'] :: Rows split into 6 clusters; headline column temperature shifts by 632.61 between clusters (Greedy deterministic partition; scores are marginal-likelihood values, not posterior probabilities.).
concentration ['temperature'] :: 13 cells in temperature have log predictive density below -9.9 nats under their cluster (lowest 999.9 at row 30; Greedy deterministic partition; scores are marginal-likelihood values, not posterior probabilities.).


## Configuration

Row caps follow the budget (standard 20,000 rows, deep 50,000) and the
`sample_rows` override; search decisions use a 2,000-row subsample and
final scores use the full sample. At most 64 eligible columns (first
in table order; the rest are recorded as dropped) and at most 8
clusters per view. The score penalty is gamma = 2.0 per cluster and
per view; the anomaly threshold is 10 nats with a cap of 20 cells per
column; a cluster needs 30 rows before it can carry a degenerate
finding. A work budget of 300 million cell visits stops search
gracefully on pathological tables and records the hit in metrics.

Continuous columns (numbers, dates, datetimes as epoch values) are
standardized by median and IQR and scored with a
normal-inverse-gamma marginal; categorical columns (strings,
booleans) keep their 32 most frequent levels with the tail pooled
into an explicit `other` level and are scored with a symmetric
Dirichlet-multinomial marginal. Nulls are excluded from scores and
their counts recorded.


## Limits, stated

- Greedy search finds local optima, not the best partition. Merge and
  split order plus tie-breaks make it deterministic, not optimal.
- Merge semantics: a merge rewards aligned structure (shared regimes,
  shared clustering), not dependence in general. Discrete columns
  whose singleton clusterings already reach purity will not merge.
  Whether to strengthen the criterion with a dependence-aware term is
  open.
- Noise splitting: the criterion splits pure-noise columns too, so
  cluster counts and regime findings on noisy columns are not, on
  their own, evidence of regimes.
- Nulls are excluded by design; missingness-driven structure is
  invisible to this pass.
- Rows are treated as an unordered set within a view. Time order,
  gaps, and cadence stay with the dates pass; a stuck run (consecutive
  equal values) is order structure and belongs to run-length
  detection, not to this pass.
- Gamma = 2.0 is a chosen default; sensitivity to it is untested.
- Categorical tail pooling loses rare-level structure; datetimes as
  continuous values lose calendar structure.


## Validation

On the sensor-iot case (50,000 rows; regime shift at row 25000,
15 spikes at 999.9), the pass recovers the regime boundary at row
24997, flags all 15 spikes at full scope (7 of 7 inside the standard
20,000-row sample), and produces byte-identical reports across runs,
in 0.7 seconds at standard budget (debug build). The stuck run in
that case is not detected by this pass, for the reason above: the
stuck value is about 6% of its regime cluster.
